# Main study: Base, RAG, CAG and QLoRA on TAT-QA

One notebook for the whole experiment. Run the cells **from top to bottom**.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

**Good to know**
- Every model and strategy saves to its **own file** (`results/raw/<model>__<strategy>.csv`). Nothing is overwritten.
- Every run cell **continues where it stopped**. Running a finished cell again just reports *0 to run*.
- **If Colab disconnects:** run *Setup A, B, C* again, then carry on from the cell you were on. Results you saved with *Step 7* come back automatically with the code.
- Do **not** use File → Save a copy in GitHub. Use *Step 7* to save results.

In [ ]:
# You should see 'Tesla T4'. If you see an error instead, there is NO GPU:
# Runtime -> Change runtime type -> T4 GPU -> Save, then run this cell again.
!nvidia-smi
import torch
print('GPU ready:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the runtime first')

## Setup A. Get the code and the results saved so far
Paste your GitHub token when asked (it starts with `ghp_`). It is hidden and not stored.

In [ ]:
import getpass, os
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
gh_token = getpass.getpass("GitHub token: ")
if not os.path.exists("/content/project"):
    !git clone -q https://{gh_token}@{REPO} /content/project
    !git -C /content/project remote set-url origin https://{REPO}
else:
    !git -C /content/project pull -q --rebase --autostash -X theirs https://{gh_token}@{REPO} main
del gh_token
%cd /content/project
!git log --oneline -1
!ls results/raw

## Setup B. Install packages and test the code (about 4 minutes)
The last line should say **31 passed**.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pytest -q 2>&1 | tail -2

## Setup C. Log in to Hugging Face
Paste your Hugging Face **Read** token (it starts with `hf_`). Llama needs the access you already have; Mistral and Qwen are open to everyone.

In [ ]:
from huggingface_hub import login
login()

## Step 1. Prepare the data (1 minute)
Downloads TAT-QA and builds the files. The official splits are kept:
- **train** → used only to train QLoRA
- **dev** → used only for the checks in Step 2
- **test** → 504 questions used for all final results

The test report extracts are put into 3 groups ("blocks") of 28. Each block is one small knowledge base: RAG searches it, CAG preloads it.

In [ ]:
!python scripts/01_prepare_tatqa.py

## Step 2. Check the setup (about 3 minutes)
**A.** How often does search find the right report extract? (needs to be above 60%, ideally above 80%)

**B.** Does every block fit in each model's memory limit?

Both must say **GOOD / OK** before you continue. If not, stop and ask.

In [ ]:
!python scripts/02_check_setup.py

## Step 3. Smoke test (about 10 minutes)
30 questions through Base, RAG and CAG on Llama-3.1-8B, to confirm everything runs. These 30 answers are kept, so the full run in Step 4 does not repeat them.

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy base rag cag --limit 30

In [ ]:
# Look at a few answers from each strategy.
import pandas as pd
pd.set_option('display.max_colwidth', 70)
import os
for s in ['base', 'rag', 'cag']:
    path = f'results/raw/llama8b__{s}.csv'
    if not os.path.exists(path):
        print(f'{s.upper()}: no results yet - the cell above did not finish. Read its last lines for the reason.')
        continue
    d = pd.read_csv(path)
    print(f"\n{s.upper()}: {d.correct.mean():.0%} correct, {d.hallucinated.mean():.0%} wrong answers, "
          f"{d.abstained.mean():.0%} said I don't know, median {d.latency_ms.median()/1000:.1f}s")
    display(d[['question', 'gold', 'final_answer', 'correct']].head(6))

## Step 4. Base, RAG and CAG: one cell per model
Each cell loads the model once and runs the three strategies on all 504 test questions (roughly 1.5 to 2 hours per model on a T4). Run **Step 7** after each model to save the results.

### 4a. Llama-3.1-8B

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy base rag cag

### 4b. Mistral-7B

In [ ]:
!python scripts/03_run_eval.py --model mistral7b --strategy base rag cag

### 4c. Qwen2.5-7B

In [ ]:
!python scripts/03_run_eval.py --model qwen7b --strategy base rag cag

## Step 5. QLoRA: one cell per model
Each cell does three things:
1. **Trains** small adapter layers on 1,202 TAT-QA *training* questions (roughly 1 to 1.5 hours on a T4). If the adapter already exists, this is skipped.
2. **Tests** the fine-tuned model on the 504 test questions, with the same documents RAG finds.
3. **Extra check:** tests it again with **no documents**, to show what fine-tuning alone can do.

Run **Step 7** after each model: it also saves the trained adapter.

### 5a. Llama-3.1-8B

In [ ]:
!python scripts/04_train_qlora.py --model llama8b
!python scripts/03_run_eval.py --model llama8b --strategy qlora --adapter adapters/llama8b_r16_s42
!python scripts/03_run_eval.py --model llama8b --strategy qlora --adapter adapters/llama8b_r16_s42 --no-docs

### 5b. Mistral-7B

In [ ]:
!python scripts/04_train_qlora.py --model mistral7b
!python scripts/03_run_eval.py --model mistral7b --strategy qlora --adapter adapters/mistral7b_r16_s42
!python scripts/03_run_eval.py --model mistral7b --strategy qlora --adapter adapters/mistral7b_r16_s42 --no-docs

### 5c. Qwen2.5-7B

In [ ]:
!python scripts/04_train_qlora.py --model qwen7b
!python scripts/03_run_eval.py --model qwen7b --strategy qlora --adapter adapters/qwen7b_r16_s42
!python scripts/03_run_eval.py --model qwen7b --strategy qlora --adapter adapters/qwen7b_r16_s42 --no-docs

## Step 6. Results
Run this at any time. It uses whatever has finished so far: the summary table, the statistical tests (once a model has two or more strategies), the TOPSIS ranking and the charts.

In [ ]:
!python scripts/05_analyze.py
!python scripts/06_plot_results.py
import pandas as pd
pd.read_csv('results/summary.csv')

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=900))

## Step 7. Save everything to GitHub
Run this **after every finished model** (it is safe to run as often as you like). It saves the results, charts and trained adapters, so a new Colab session can continue from them.

In [ ]:
import getpass, datetime
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
tok = getpass.getpass("GitHub token: ")
!git config user.name "Faheem Jabbar"
!git config user.email "faheemjabbar39@gmail.com"
!git add results
!test -d adapters && git add adapters
msg = f"Main study results ({datetime.date.today()})"
!git commit -q -m "{msg}" || echo "nothing new to save"
!git pull -q --rebase --autostash -X theirs https://{tok}@{REPO} main
!git push -q https://{tok}@{REPO} HEAD:main && echo "Saved to GitHub"
del tok